# Reconhecimento de Objetos com YOLOv8 — PyTorch Puro

Este notebook implementa a detecção de objetos usando a arquitetura **YOLOv8-nano** reconstruída do zero em PyTorch.  
Os pesos pré-treinados são carregados diretamente do arquivo `.pt` oficial, **sem usar nenhuma função de inferência da biblioteca Ultralytics**.

### O que você precisa implementar:
- `manual_iou()` — cálculo da Intersecção Sobre a União
- `manual_nms()` — Supressão Não-Maximal

O restante (arquitetura, carregamento de pesos, decoder DFL, pré/pós-processamento) já está implementado.

## 0. Setup — Monte o Drive e instale dependências

In [1]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: mount failed

In [ ]:
# Instala apenas o necessário — sem ultralytics!
!pip install -q torch torchvision pillow matplotlib numpy

In [ ]:
# Baixa o arquivo de pesos YOLOv8-nano diretamente do GitHub da Ultralytics
# Isso só baixa o .pt (pesos), não instala a biblioteca
import urllib.request, os

WEIGHTS_URL = "https://github.com/ultralytics/assets/releases/download/v8.2.0/yolov8n.pt"
WEIGHTS_PATH = "yolov8n.pt"

if not os.path.exists(WEIGHTS_PATH):
    print("Baixando yolov8n.pt...")
    urllib.request.urlretrieve(WEIGHTS_URL, WEIGHTS_PATH)
    print(f"Download concluído: {os.path.getsize(WEIGHTS_PATH)/1e6:.1f} MB")
else:
    print("Arquivo de pesos já existe.")

Baixando yolov8n.pt...
Download concluído: 6.5 MB


In [ ]:
# Baixa o arquivo de classes COCO
COCO_URL = "https://raw.githubusercontent.com/pjreddie/darknet/master/data/coco.names"
if not os.path.exists("coco.names"):
    urllib.request.urlretrieve(COCO_URL, "coco.names")
    print("coco.names baixado.")

coco.names baixado.


## 1. Arquitetura YOLOv8 — Implementação do Zero

O YOLOv8 é composto por três partes:
1. **Backbone CSPDarknet**: extrai features em múltiplas escalas usando blocos `C2f` (Cross-Stage Partial com 2 convoluções de fusão)
2. **Neck FPN+PAN**: combina features de diferentes escalas (Feature Pyramid Network + Path Aggregation Network)
3. **Detection Head**: para cada escala, prevê caixas usando DFL (Distribution Focal Loss) e classes separadamente

### Diferença principal em relação ao YOLOv3:
- **C2f** substitui os blocos residuais do Darknet-53
- **DFL head** em vez de saída direta: o modelo prevê uma *distribuição de probabilidade* sobre posições discretas para cada borda da caixa, e o valor final é a esperança dessa distribuição
- **Saída desacoplada**: confiança de objeto e classe são previstas em ramos separados

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageDraw, ImageFont
import colorsys, random, math

# ===========================================================
# BLOCOS FUNDAMENTAIS DA ARQUITETURA
# ===========================================================

class Conv(nn.Module):
    """
    Bloco base do YOLOv8: Conv2d -> BatchNorm2d -> SiLU
    Nota: O v8 usa SiLU (Sigmoid Linear Unit) em vez do LeakyReLU do v3.
    SiLU(x) = x * sigmoid(x) — mais suave e tem gradientes melhores.
    """
    def __init__(self, in_c, out_c, k=1, s=1, p=None, g=1):
        super().__init__()
        # Padding automático para manter dimensão espacial quando k>1
        if p is None:
            p = k // 2
        self.conv = nn.Conv2d(in_c, out_c, k, s, p, groups=g, bias=False)
        self.bn   = nn.BatchNorm2d(out_c, eps=1e-3, momentum=0.03)
        self.act  = nn.SiLU(inplace=True)

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))


class Bottleneck(nn.Module):
    """
    Bottleneck do YOLOv8: duas convoluções 3x3 com conexão residual opcional.
    Diferente do v3 (que usava 1x1 -> 3x3), o v8 usa 3x3 -> 3x3.
    """
    def __init__(self, in_c, out_c, shortcut=True, e=0.5):
        super().__init__()
        hidden = int(out_c * e)
        self.cv1 = Conv(in_c,   hidden, 3, 1)
        self.cv2 = Conv(hidden, out_c,  3, 1)
        self.add = shortcut and (in_c == out_c)

    def forward(self, x):
        return x + self.cv2(self.cv1(x)) if self.add else self.cv2(self.cv1(x))


class C2f(nn.Module):
    """
    C2f (Cross-Stage Partial com 2 fusões) — bloco central do YOLOv8.

    Funcionamento:
    1. Divide o tensor de entrada em dois ramos após uma conv 1x1
    2. Um ramo passa por N Bottlenecks sequenciais; o resultado
       de cada bottleneck é guardado
    3. Todos os tensores intermediários são concatenados
    4. Uma conv 1x1 final funde tudo

    Isso é mais eficiente que o CSP do v5 e permite gradientes
    fluírem por caminhos múltiplos.
    """
    def __init__(self, in_c, out_c, n=1, shortcut=True, e=0.5):
        super().__init__()
        hidden = int(out_c * e)
        self.cv1 = Conv(in_c,       2 * hidden, 1, 1)
        self.cv2 = Conv((2 + n) * hidden, out_c, 1, 1)
        self.m   = nn.ModuleList(
            [Bottleneck(hidden, hidden, shortcut, e=1.0) for _ in range(n)]
        )

    def forward(self, x):
        # Divide em dois: y[0] e y[1] são os dois ramos iniciais
        y = list(self.cv1(x).chunk(2, dim=1))
        # Cada bottleneck recebe a saída do anterior
        y.extend(m(y[-1]) for m in self.m)
        # Concatena todos e funde
        return self.cv2(torch.cat(y, dim=1))


class SPPF(nn.Module):
    """
    Spatial Pyramid Pooling — Fast (SPPF).
    Captura contexto global aplicando MaxPool de tamanho k três vezes
    sequencialmente (equivalente a pools de k, 2k-1, 3k-1 em paralelo,
    mas muito mais rápido).
    Localizado no fim do backbone antes do neck.
    """
    def __init__(self, in_c, out_c, k=5):
        super().__init__()
        hidden = in_c // 2
        self.cv1 = Conv(in_c,      hidden, 1, 1)
        self.cv2 = Conv(hidden * 4, out_c, 1, 1)
        self.m   = nn.MaxPool2d(k, stride=1, padding=k // 2)

    def forward(self, x):
        x  = self.cv1(x)
        y1 = self.m(x)
        y2 = self.m(y1)
        return self.cv2(torch.cat([x, y1, y2, self.m(y2)], dim=1))


class DFL(nn.Module):
    """
    Distribution Focal Loss — camada de decodificação das caixas.

    No YOLOv8, em vez de prever (dx, dy, dw, dh) diretamente,
    o modelo prevê uma distribuição de probabilidade discreta
    sobre `reg_max` posições para cada uma das 4 bordas da caixa.

    O valor final de cada borda é a esperança (valor esperado)
    dessa distribuição:
        borda = sum(i * softmax(logits)[i] for i in range(reg_max))

    Isso é equivalente a uma conv 1x1 com pesos fixos [0, 1, ..., reg_max-1]
    aplicada após o softmax — exatamente o que fazemos aqui.
    """
    def __init__(self, reg_max=16):
        super().__init__()
        self.reg_max = reg_max
        # Pesos fixos: [0, 1, 2, ..., reg_max-1] — não treinados!
        self.conv = nn.Conv2d(reg_max, 1, 1, bias=False)
        self.conv.weight.data = torch.arange(
            reg_max, dtype=torch.float32
        ).reshape(1, reg_max, 1, 1)
        self.conv.weight.requires_grad_(False)

    def forward(self, x):
        # x: (B, 4*reg_max, H, W)
        B, C, H, W = x.shape
        # Separa as 4 distribuições (uma por borda)
        x = x.view(B, 4, self.reg_max, H * W)
        x = x.permute(0, 3, 1, 2)          # (B, H*W, 4, reg_max)
        x = F.softmax(x, dim=-1)            # distribuição de probabilidade
        x = x.permute(0, 2, 3, 1)          # (B, 4, reg_max, H*W)
        x = x.reshape(B * 4, self.reg_max, 1, H * W)
        # Conv com pesos fixos = valor esperado da distribuição
        return self.conv(x).reshape(B, 4, H * W)

In [ ]:
# ===========================================================
# ARQUITETURA COMPLETA DO YOLOv8-nano
# ===========================================================

class DetectHead(nn.Module):
    """
    Cabeçalho de detecção do YOLOv8.

    Diferença crítica em relação ao YOLOv3:
    - Ramo de regressão (caixas) e ramo de classificação são SEPARADOS
    - Não há 'objectness score' — a confiança final é só a probabilidade de classe
    - Regressão usa DFL em vez de previsão direta de coordenadas
    """
    def __init__(self, num_classes=80, ch=()):  # ch = canais de entrada de cada escala
        super().__init__()
        self.nc       = num_classes
        self.reg_max  = 16             # resolução da distribuição DFL
        self.no       = num_classes + self.reg_max * 4
        self.nl       = len(ch)        # número de escalas (3)
        self.stride   = torch.zeros(self.nl)

        # Canais do ramo de regressão: max(16, reg_max*4, ch[i]//4)
        reg_ch = max(16, self.reg_max * 4, ch[0] // 4)
        # Canais do ramo de classificação: max(num_classes, ch[i])
        cls_ch = max(num_classes, ch[0])

        # Um par de convoluções para cada escala, para cada ramo
        self.cv2 = nn.ModuleList(  # ramo de regressão (caixas)
            nn.Sequential(Conv(c, reg_ch, 3), Conv(reg_ch, reg_ch, 3),
                          nn.Conv2d(reg_ch, 4 * self.reg_max, 1)) for c in ch
        )
        self.cv3 = nn.ModuleList(  # ramo de classificação
            nn.Sequential(Conv(c, cls_ch, 3), Conv(cls_ch, cls_ch, 3),
                          nn.Conv2d(cls_ch, num_classes, 1)) for c in ch
        )
        self.dfl = DFL(self.reg_max)

    def forward(self, feats):
        # Retorna as features brutas de cada escala durante treinamento/exportação
        # Durante inferência, decodificamos fora (em decode_predictions)
        outputs = []
        for i, x in enumerate(feats):
            box_feat = self.cv2[i](x)   # (B, 4*reg_max, H, W)
            cls_feat = self.cv3[i](x)   # (B, nc, H, W)
            outputs.append((box_feat, cls_feat))
        return outputs


class YOLOv8n(nn.Module):
    """
    YOLOv8-nano implementado do zero.

    Fatores de escala do YOLOv8-nano:
      depth_multiple = 0.33  (número de repetições dos blocos C2f)
      width_multiple = 0.25  (número de canais)

    Arquitetura:
    ┌─────────────────────────────────────────────────────┐
    │  BACKBONE (extração de features)                     │
    │  Conv → C2f → C2f → C2f → C2f → SPPF               │
    │         P3        P4         P5                      │
    ├─────────────────────────────────────────────────────┤
    │  NECK (Feature Pyramid Network + Path Aggregation)   │
    │  Upsample P5→P4→P3 (top-down) + C2f                 │
    │  Conv P3→P4→P5 (bottom-up) + C2f                    │
    ├─────────────────────────────────────────────────────┤
    │  HEAD (detecção em 3 escalas)                        │
    │  [P3_out, P4_out, P5_out] → DetectHead              │
    └─────────────────────────────────────────────────────┘
    """
    def __init__(self, num_classes=80):
        super().__init__()
        # Para nano: width=0.25, depth=0.33
        # Canais base: [64, 128, 256, 512] escalados por 0.25 = [16, 32, 64, 128]
        # Repetições base: [3, 6, 6, 3] escaladas por 0.33 = [1, 2, 2, 1]

        # --- BACKBONE ---
        self.model = nn.Sequential(
            # Layer 0: entrada 3ch -> 16ch
            Conv(3, 16, 3, 2),               # /2  -> 320x320
            # Layer 1
            Conv(16, 32, 3, 2),              # /4  -> 160x160
            # Layer 2
            C2f(32, 32, n=1, shortcut=True), # 160x160
            # Layer 3
            Conv(32, 64, 3, 2),              # /8  -> 80x80  (P3)
            # Layer 4
            C2f(64, 64, n=2, shortcut=True), # 80x80
            # Layer 5
            Conv(64, 128, 3, 2),             # /16 -> 40x40  (P4)
            # Layer 6
            C2f(128, 128, n=2, shortcut=True),# 40x40
            # Layer 7
            Conv(128, 256, 3, 2),            # /32 -> 20x20  (P5)
            # Layer 8
            C2f(256, 256, n=1, shortcut=True),# 20x20
            # Layer 9
            SPPF(256, 256, k=5),             # 20x20  (fim do backbone)
        )

        # --- NECK (top-down: P5->P4->P3) ---
        self.up1  = nn.Upsample(scale_factor=2, mode='nearest')  # P5 20->40
        self.c2f_p4_td = C2f(256 + 128, 128, n=1, shortcut=False)

        self.up2  = nn.Upsample(scale_factor=2, mode='nearest')  # P4 40->80
        self.c2f_p3_td = C2f(128 + 64, 64, n=1, shortcut=False)

        # --- NECK (bottom-up: P3->P4->P5) ---
        self.down1     = Conv(64, 64, 3, 2)                       # P3 80->40
        self.c2f_p4_bu = C2f(64 + 128, 128, n=1, shortcut=False)

        self.down2     = Conv(128, 128, 3, 2)                     # P4 40->20
        self.c2f_p5_bu = C2f(128 + 256, 256, n=1, shortcut=False)

        # --- HEAD ---
        self.head = DetectHead(num_classes, ch=(64, 128, 256))

    def forward(self, x):
        # Backbone — extrai P3, P4, P5
        for i, layer in enumerate(self.model):
            x = layer(x)
            if i == 4:  p3 = x   # 80x80, 64ch
            if i == 6:  p4 = x   # 40x40, 128ch
            if i == 9:  p5 = x   # 20x20, 256ch  (após SPPF)

        # Neck top-down (FPN)
        x      = self.up1(p5)                           # 40x40
        x      = self.c2f_p4_td(torch.cat([x, p4], 1)) # funde com P4
        p4_out = x                                       # salva para bottom-up

        x      = self.up2(x)                            # 80x80
        x      = self.c2f_p3_td(torch.cat([x, p3], 1)) # funde com P3
        p3_out = x                                       # saída P3 do neck

        # Neck bottom-up (PAN)
        x      = self.down1(p3_out)                          # 40x40
        x      = self.c2f_p4_bu(torch.cat([x, p4_out], 1))  # funde com P4
        p4_fin = x

        x      = self.down2(p4_fin)                          # 20x20
        p5_fin = self.c2f_p5_bu(torch.cat([x, p5], 1))      # funde com P5

        # Head: 3 escalas
        return self.head([p3_out, p4_fin, p5_fin])

## 2. Carregamento dos Pesos do YOLOv8

O arquivo `.pt` salvo pela Ultralytics contém um modelo Python completo (não apenas pesos).  
Vamos inspecioná-lo e mapear os pesos para a nossa arquitetura manualmente.

In [ ]:
def inspecionar_pesos(caminho_pt):
    """
    Carrega e inspeciona o arquivo .pt sem usar a biblioteca Ultralytics.
    O arquivo contém um dicionário com o modelo completo.
    """
    ckpt = torch.load(caminho_pt, map_location='cpu', weights_only=False)
    print("Chaves no checkpoint:", list(ckpt.keys()))

    # O modelo está em ckpt['model'] — é o objeto Python da Ultralytics
    # Mas podemos extrair o state_dict dele sem executar nenhuma inferência
    modelo_ultra = ckpt['model']
    state = modelo_ultra.state_dict()

    print(f"\nTotal de tensores no state_dict: {len(state)}")
    print("\nPrimeiras 20 chaves:")
    for k in list(state.keys())[:20]:
        print(f"  {k:60s} {tuple(state[k].shape)}")
    return state

state_ultra = inspecionar_pesos("yolov8n.pt")

In [ ]:
def carregar_pesos_yolov8(caminho_pt, modelo):
    """
    Carrega os pesos do .pt oficial nos nossos módulos PyTorch.

    A Ultralytics numera as camadas de 0 a 22 no state_dict.
    Fazemos o mapeamento para os nomes do nosso modelo manualmente.
    """
    ckpt  = torch.load(caminho_pt, map_location='cpu', weights_only=False)
    src   = ckpt['model'].state_dict()

    # Mapeamento: nome_ultralytics -> nome_no_nosso_modelo
    # As camadas 0-9 são o backbone (model[0] a model[9])
    # As camadas 10-22 são neck + head
    mapa = {
        # BACKBONE
        'model.0.':  'model.0.',
        'model.1.':  'model.1.',
        'model.2.':  'model.2.',
        'model.3.':  'model.3.',
        'model.4.':  'model.4.',
        'model.5.':  'model.5.',
        'model.6.':  'model.6.',
        'model.7.':  'model.7.',
        'model.8.':  'model.8.',
        'model.9.':  'model.9.',
        # NECK top-down
        'model.12.': 'c2f_p4_td.',
        'model.15.': 'c2f_p3_td.',
        # NECK bottom-up
        'model.16.': 'down1.',
        'model.18.': 'c2f_p4_bu.',
        'model.19.': 'down2.',
        'model.21.': 'c2f_p5_bu.',
        # HEAD
        'model.22.cv2.': 'head.cv2.',
        'model.22.cv3.': 'head.cv3.',
        'model.22.dfl.': 'head.dfl.',
    }

    dst = {}
    nao_mapeado = []
    for k_src, v in src.items():
        k_dst = None
        for prefixo_src, prefixo_dst in mapa.items():
            if k_src.startswith(prefixo_src):
                k_dst = k_src.replace(prefixo_src, prefixo_dst, 1)
                break
        if k_dst is not None:
            dst[k_dst] = v
        else:
            nao_mapeado.append(k_src)

    resultado = modelo.load_state_dict(dst, strict=False)
    print(f"Pesos carregados: {len(dst)}/{len(src)}")
    if resultado.missing_keys:
        print(f"  Faltando no nosso modelo ({len(resultado.missing_keys)}):")
        for k in resultado.missing_keys[:10]:
            print(f"    {k}")
    if resultado.unexpected_keys:
        print(f"  Inesperados ({len(resultado.unexpected_keys)}): (geralmente são strides/anchors internos)")
    if nao_mapeado:
        print(f"  Não mapeados do source ({len(nao_mapeado)}):")
        for k in nao_mapeado[:5]:
            print(f"    {k}")
    return modelo


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo: {device}")

modelo = YOLOv8n(num_classes=80).to(device)
modelo = carregar_pesos_yolov8('yolov8n.pt', modelo)
modelo.eval()
print("\nModelo pronto!")

## 3. Pré-processamento de Imagens

In [ ]:
def ler_classes(caminho):
    with open(caminho) as f:
        return [c.strip() for c in f.readlines()]

def gerar_cores(nomes_classes):
    hsv = [(x / len(nomes_classes), 0.9, 0.9) for x in range(len(nomes_classes))]
    cores = [tuple(int(v*255) for v in colorsys.hsv_to_rgb(*h)) for h in hsv]
    random.seed(42)
    random.shuffle(cores)
    return cores

def letterbox(image, novo_tamanho=(640, 640), cor=(114, 114, 114)):
    """
    Redimensiona a imagem mantendo a proporção original.
    O espaço restante é preenchido com a cor especificada (cinza neutro).
    Isso evita distorção geométrica que prejudicaria a detecção.
    """
    iw, ih = image.size
    nw, nh = novo_tamanho
    escala = min(nw / iw, nh / ih)
    nw_r   = int(iw * escala)
    nh_r   = int(ih * escala)

    img_r = image.resize((nw_r, nh_r), Image.BICUBIC)
    canvas = Image.new('RGB', novo_tamanho, cor)
    offset_x = (novo_tamanho[0] - nw_r) // 2
    offset_y = (novo_tamanho[1] - nh_r) // 2
    canvas.paste(img_r, (offset_x, offset_y))
    return canvas, escala, offset_x, offset_y

def preprocessar(caminho_img, tamanho=640):
    """Carrega, aplica letterbox e converte para tensor normalizado."""
    img_orig = Image.open(caminho_img).convert('RGB')
    img_lb, escala, ox, oy = letterbox(img_orig, (tamanho, tamanho))

    arr = np.array(img_lb, dtype=np.float32) / 255.0
    arr = np.transpose(arr, (2, 0, 1))  # HWC -> CHW
    tensor = torch.from_numpy(arr).unsqueeze(0)  # adiciona dim batch

    return img_orig, tensor, escala, ox, oy

def reverter_caixas(boxes, escala, offset_x, offset_y, img_orig_size):
    """
    Converte coordenadas do espaço letterbox (640x640) de volta
    para o espaço da imagem original.

    boxes: tensor (N, 4) em formato (x1, y1, x2, y2), pixels do letterbox
    """
    boxes = boxes.clone().float()
    # Remove o offset das bordas cinzas
    boxes[:, [0, 2]] -= offset_x
    boxes[:, [1, 3]] -= offset_y
    # Desfaz a escala
    boxes /= escala
    # Clipa dentro dos limites da imagem original
    iw, ih = img_orig_size
    boxes[:, [0, 2]] = boxes[:, [0, 2]].clamp(0, iw)
    boxes[:, [1, 3]] = boxes[:, [1, 3]].clamp(0, ih)
    return boxes

class_names = ler_classes('coco.names')
CORES = gerar_cores(class_names)
print(f"{len(class_names)} classes carregadas. Exemplo: {class_names[:5]}")

## 4. Decoder das Saídas do YOLOv8

A saída do modelo para cada escala é um par `(box_feat, cls_feat)`.  
Precisamos converter isso para caixas absolutas em pixels.

In [ ]:
def gerar_grade(h, w, stride, device):
    """
    Gera a grade de âncoras implícitas do YOLOv8.
    No v8, não há âncoras explícitas (como no v3).
    Cada célula da grade é o ponto central de uma potencial detecção.
    """
    gy, gx = torch.meshgrid(
        torch.arange(h, device=device),
        torch.arange(w, device=device),
        indexing='ij'
    )
    # Pontos centrais de cada célula em pixels (+ 0.5 para o centro)
    grade = torch.stack([gx, gy], dim=-1).reshape(-1, 2).float()
    return (grade + 0.5) * stride


def decodificar_saidas(outputs, img_size=640, device='cpu'):
    """
    Decodifica as saídas brutas do DetectHead em caixas absolutas.

    O YOLOv8 prevê, para cada célula da grade:
      - box_feat: distribuição DFL para as 4 bordas (dist. ao centro)
      - cls_feat: logits de classe (sem objectness separado)

    A decodificação DFL:
      dist = DFL(box_feat) -> [dl, dt, dr, db] (distâncias ao centro)
      x1 = cx - dl, y1 = cy - dt
      x2 = cx + dr, y2 = cy + db
    """
    # Strides para cada escala: 8 (80x80), 16 (40x40), 32 (20x20)
    strides = [img_size // s for s in [80, 40, 20]]
    dfl = DFL(reg_max=16).to(device)

    todas_caixas  = []
    todas_classes = []

    for i, ((box_feat, cls_feat), stride) in enumerate(zip(outputs, strides)):
        B, _, H, W = box_feat.shape

        # Grade de pontos centrais
        grade = gerar_grade(H, W, stride, device)  # (H*W, 2)

        # Decodifica DFL -> distâncias (dl, dt, dr, db)
        dist = dfl(box_feat).squeeze(0)             # (4, H*W)
        dist = dist * stride                        # escala para pixels

        # Converte distâncias ao centro para coordenadas absolutas
        dl, dt, dr, db = dist[0], dist[1], dist[2], dist[3]
        cx, cy = grade[:, 0], grade[:, 1]

        x1 = (cx - dl).unsqueeze(1)
        y1 = (cy - dt).unsqueeze(1)
        x2 = (cx + dr).unsqueeze(1)
        y2 = (cy + db).unsqueeze(1)
        caixas = torch.cat([x1, y1, x2, y2], dim=1)   # (H*W, 4)

        # Classes: sigmoid (v8 não usa softmax nas classes)
        classes = torch.sigmoid(cls_feat.squeeze(0).reshape(80, -1).T)  # (H*W, 80)

        todas_caixas.append(caixas)
        todas_classes.append(classes)

    todas_caixas  = torch.cat(todas_caixas,  dim=0)   # (total_anchors, 4)
    todas_classes = torch.cat(todas_classes, dim=0)   # (total_anchors, 80)
    return todas_caixas, todas_classes

## 5. ⚠️ Implemente Aqui: IoU e NMS

Estas são as duas funções que você deve implementar.  
Os esqueletos e as dicas estão abaixo.

In [ ]:
def manual_iou(box, boxes):
    """
    Calcula o IoU (Intersection over Union) entre uma caixa e um conjunto de caixas.

    Parâmetros:
        box   : tensor (4,)   — uma caixa no formato (x1, y1, x2, y2)
        boxes : tensor (N, 4) — N caixas no mesmo formato

    Retorna:
        iou   : tensor (N,)   — IoU de `box` com cada caixa em `boxes`

    Dicas:
      - A interseção de dois retângulos é outro retângulo.
      - Coordenadas da interseção:
            x1_inter = max(box[0], boxes[:, 0])
            y1_inter = max(box[1], boxes[:, 1])
            x2_inter = min(box[2], boxes[:, 2])
            y2_inter = min(box[3], boxes[:, 3])
      - Área da interseção: max(0, x2-x1) * max(0, y2-y1)
            (use .clamp(min=0) para vetorizar)
      - Área de cada caixa: (x2 - x1) * (y2 - y1)
      - IoU = intersecao / (area_box + area_boxes - intersecao)
    """
    # ===================== IMPLEMENTE AQUI =====================
    raise NotImplementedError("Implemente manual_iou!")
    # ===========================================================


def manual_nms(boxes, scores, iou_threshold=0.45):
    """
    Non-Maximum Suppression: remove detecções redundantes.

    Algoritmo:
      1. Ordene as detecções por score (maior primeiro)
      2. Pegue a detecção com maior score e adicione ao resultado
      3. Calcule o IoU dessa detecção com todas as restantes
      4. Remova as detecções com IoU > iou_threshold (são redundantes)
      5. Repita com as detecções restantes até não sobrar nenhuma

    Parâmetros:
        boxes         : tensor (N, 4) — caixas (x1, y1, x2, y2)
        scores        : tensor (N,)   — score de cada caixa
        iou_threshold : float         — limiar de IoU para supressão

    Retorna:
        keep : tensor (K,) — índices das caixas mantidas (K <= N)

    Dicas:
      - Use torch.argsort(scores, descending=True) para ordenar
      - Mantenha uma lista `keep` de índices selecionados
      - Use manual_iou() que você implementou acima
      - Remova da lista os índices com IoU > iou_threshold
    """
    # ===================== IMPLEMENTE AQUI =====================
    raise NotImplementedError("Implemente manual_nms!")
    # ===========================================================

## 6. Inferência e Visualização

In [ ]:
def detectar(caminho_img, modelo, class_names, device,
             score_threshold=0.25, iou_threshold=0.45,
             img_size=640, max_deteccoes=100):
    """
    Pipeline completo de detecção:
      1. Pré-processamento (letterbox + tensor)
      2. Inferência
      3. Decodificação DFL
      4. Filtragem por score
      5. NMS manual
      6. Reversão de coordenadas
      7. Visualização
    """
    print(f"\nProcessando: {caminho_img}")

    # 1. Pré-processamento
    img_orig, tensor, escala, ox, oy = preprocessar(caminho_img, img_size)
    tensor = tensor.to(device)

    # 2. Inferência
    modelo.eval()
    with torch.no_grad():
        outputs = modelo(tensor)  # lista de 3 escalas

    # 3. Decodificação
    caixas, classes = decodificar_saidas(outputs, img_size, device)

    # 4. Filtragem por score de confiança
    scores_max, classes_idx = classes.max(dim=1)
    mascara = scores_max >= score_threshold
    caixas      = caixas[mascara]
    scores_max  = scores_max[mascara]
    classes_idx = classes_idx[mascara]

    print(f"  Detecções antes do NMS: {len(caixas)}")

    if len(caixas) == 0:
        print("  Nenhum objeto detectado.")
        plt.figure(figsize=(10, 8))
        plt.imshow(img_orig)
        plt.axis('off')
        plt.title("Nenhum objeto detectado")
        plt.show()
        return

    # 5. NMS por classe (aplica NMS separadamente para cada classe)
    keep_final = []
    for cls in classes_idx.unique():
        mascara_cls = classes_idx == cls
        idx_cls     = mascara_cls.nonzero(as_tuple=True)[0]
        keep_cls    = manual_nms(
            caixas[idx_cls], scores_max[idx_cls], iou_threshold
        )
        keep_final.append(idx_cls[keep_cls])

    keep_final  = torch.cat(keep_final)
    # Reordena por score e limita ao máximo
    ordem       = scores_max[keep_final].argsort(descending=True)[:max_deteccoes]
    keep_final  = keep_final[ordem]

    caixas_finais  = caixas[keep_final]
    scores_finais  = scores_max[keep_final]
    classes_finais = classes_idx[keep_final]

    # 6. Reverter para coordenadas da imagem original
    caixas_orig = reverter_caixas(caixas_finais, escala, ox, oy, img_orig.size)

    n = len(caixas_finais)
    print(f"  Detecções após NMS: {n}")
    print(f"  Objetos detectados:")
    for i in range(n):
        nome  = class_names[classes_finais[i].item()]
        conf  = scores_finais[i].item()
        print(f"    [{i+1:2d}] {nome:20s}  confiança: {conf:.3f}")

    # 7. Visualização
    img_draw = img_orig.copy()
    draw     = ImageDraw.Draw(img_draw)
    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 14)
    except:
        font = ImageFont.load_default()

    espessura = max(2, (img_orig.size[0] + img_orig.size[1]) // 500)

    for i in range(n):
        x1, y1, x2, y2 = caixas_orig[i].cpu().tolist()
        cls_id = classes_finais[i].item()
        score  = scores_finais[i].item()
        nome   = class_names[cls_id]
        cor    = CORES[cls_id]

        # Bounding box
        for t in range(espessura):
            draw.rectangle([x1+t, y1+t, x2-t, y2-t], outline=cor)

        # Label
        label = f"{nome}: {score:.2f}"
        bbox_txt = draw.textbbox((x1, y1), label, font=font)
        draw.rectangle(bbox_txt, fill=cor)
        draw.text((x1, y1), label, fill=(255, 255, 255), font=font)

    plt.figure(figsize=(12, 10))
    plt.imshow(img_draw)
    plt.axis('off')
    plt.title(f"{n} objeto(s) detectado(s) — score≥{score_threshold}, IoU≤{iou_threshold}")
    plt.tight_layout()
    plt.show()
    return caixas_orig, scores_finais, classes_finais

## 7. Teste nas Imagens

In [ ]:
# =====================================================
# Rode uma imagem de teste
# Altere o caminho conforme necessário
# =====================================================
resultado = detectar(
    "images/dog.jpg",       # <- altere para a sua imagem
    modelo,
    class_names,
    device,
    score_threshold=0.25,
    iou_threshold=0.45
)

## 8. Comparação: Efeito dos Limiares de NMS e IoU

Experimente diferentes valores e observe o impacto nas detecções.

In [ ]:
# =====================================================
# Comparação sistemática dos limiares
# =====================================================
configs = [
    {"score_threshold": 0.10, "iou_threshold": 0.30, "desc": "Score baixo, IoU baixo (muitas detecções, poucas suprimidas)"},
    {"score_threshold": 0.25, "iou_threshold": 0.45, "desc": "Configuração padrão YOLOv8"},
    {"score_threshold": 0.50, "iou_threshold": 0.60, "desc": "Score alto, IoU alto (poucas e mais confiantes)"},
    {"score_threshold": 0.70, "iou_threshold": 0.45, "desc": "Score muito alto (só objetos muito óbvios)"},
]

IMAGEM_TESTE = "images/dog.jpg"  # <- altere conforme necessário

fig, axes = plt.subplots(2, 2, figsize=(16, 14))

for ax, cfg in zip(axes.flat, configs):
    img_orig, tensor, escala, ox, oy = preprocessar(IMAGEM_TESTE)
    tensor = tensor.to(device)

    with torch.no_grad():
        outputs = modelo(tensor)

    caixas, classes = decodificar_saidas(outputs, 640, device)
    scores_max, classes_idx = classes.max(dim=1)
    mascara = scores_max >= cfg["score_threshold"]
    caixas = caixas[mascara]
    scores_max = scores_max[mascara]
    classes_idx = classes_idx[mascara]

    if len(caixas) > 0:
        keep_final = []
        for cls in classes_idx.unique():
            m = classes_idx == cls
            idx = m.nonzero(as_tuple=True)[0]
            keep = manual_nms(caixas[idx], scores_max[idx], cfg["iou_threshold"])
            keep_final.append(idx[keep])
        keep_final = torch.cat(keep_final)
        caixas_finais = caixas[keep_final]
        scores_finais = scores_max[keep_final]
        classes_finais = classes_idx[keep_final]
        caixas_orig = reverter_caixas(caixas_finais, escala, ox, oy, img_orig.size)

        img_draw = img_orig.copy()
        draw = ImageDraw.Draw(img_draw)
        try:
            font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 13)
        except:
            font = ImageFont.load_default()

        for i in range(len(caixas_finais)):
            x1, y1, x2, y2 = caixas_orig[i].cpu().tolist()
            cls_id = classes_finais[i].item()
            cor = CORES[cls_id]
            label = f"{class_names[cls_id]}: {scores_finais[i]:.2f}"
            draw.rectangle([x1, y1, x2, y2], outline=cor, width=2)
            draw.text((x1+2, y1+2), label, fill=cor, font=font)
        n = len(caixas_finais)
    else:
        img_draw = img_orig
        n = 0

    ax.imshow(img_draw)
    ax.axis('off')
    ax.set_title(f"{cfg['desc']}\n({n} detecções)", fontsize=9)

plt.suptitle("Efeito dos Limiares de Score e IoU", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Verificação da Arquitetura

In [ ]:
# Exibe um resumo da arquitetura do modelo
total_params = sum(p.numel() for p in modelo.parameters())
treinaveis   = sum(p.numel() for p in modelo.parameters() if p.requires_grad)

print("=" * 60)
print("ARQUITETURA YOLOv8-nano (implementação manual)")
print("=" * 60)

print("\n[ BACKBONE ]")
for i, layer in enumerate(modelo.model):
    n = sum(p.numel() for p in layer.parameters())
    print(f"  Layer {i:2d}: {type(layer).__name__:10s}  params: {n:,}")

print("\n[ NECK ]")
for nome, mod in [
    ('up1 (Upsample 20->40)', modelo.up1),
    ('c2f_p4_td',             modelo.c2f_p4_td),
    ('up2 (Upsample 40->80)', modelo.up2),
    ('c2f_p3_td',             modelo.c2f_p3_td),
    ('down1 (Conv 80->40)',   modelo.down1),
    ('c2f_p4_bu',             modelo.c2f_p4_bu),
    ('down2 (Conv 40->20)',   modelo.down2),
    ('c2f_p5_bu',             modelo.c2f_p5_bu),
]:
    n = sum(p.numel() for p in mod.parameters())
    print(f"  {nome:30s}  params: {n:,}")

print("\n[ HEAD ]")
print(f"  DetectHead (3 escalas: 80x80, 40x40, 20x20)")
n = sum(p.numel() for p in modelo.head.parameters())
print(f"  params: {n:,}")

print("\n" + "=" * 60)
print(f"  Total de parâmetros:      {total_params:,}")
print(f"  Parâmetros treináveis:    {treinaveis:,}")
print("=" * 60)